# 01 — Grow one tree

This notebook is the literate-programming companion to
[`examples/grow_one_tree.py`](../examples/grow_one_tree.py). We load a YAML
configuration, run a single-tree simulation for a hundred generations
and look at the result in 3D.

## What is happening under the hood

Each generation, MechaTree runs the loop introduced in Eloy et al.,
*Wind loads and competition for light sculpt trees into self-similar structures*
([Nat Commun 8, 1014, 2017](https://doi.org/10.1038/s41467-017-00995-6)):

1. **Light interception.** Leaves are projected onto a sun grid; each one
   captures a share of the available photons.
2. **Mechanics.** A random storm wind exerts drag on the canopy; bending
   moments are propagated from twigs down to the trunk.
3. **Growth.** Each branch widens just enough to keep its peak stress
   below `safety × breaking stress`; spare resources fuel new twigs.
4. **Pruning.** Branches whose stress exceeds the breaking threshold are
   cut, along with everything they support.
5. **Reorder.** Strahler / Horton indices are recomputed.

Run this cell top-to-bottom; everything finishes in a few seconds.

In [ ]:
from pathlib import Path

from mechatree.config import load_config
from mechatree.light import Sun
from mechatree.plotting import figstyle, plot_tree_3d
from mechatree.simulate import grow_tree

figstyle.apply()  # MATLAB-style plotly template (white bg, boxed, inside ticks)

config_path = Path("../examples/forest.yaml").resolve()
cfg = load_config(config_path)
cfg.tree.twig_length, cfg.tree.cauchy, cfg.genome.safety

The `cfg` object is a typed view of the YAML — `cfg.tree`, `cfg.light`,
`cfg.forest`, `cfg.genome`. The defaults above match the reference run in
the paper: a 0.1 m twig diameter, soft cauchy stiffness ($C_y =
2\times10^{-5}$), and a safety factor of 3.0 (so every branch sits at
about $(1/3)^{3/2} \approx 19\%$ of breaking stress under nominal wind).

## Run the simulation

We sample the per-generation state through an `on_step` callback. The
callback receives the generation index and the live `PyTree`.

In [ ]:
n_generations = 100
history = []  # (gen, n_branches, n_leaves, reserve)


def on_step(gen, tree):
    if gen % 10 == 0 or gen == n_generations - 1:
        history.append(
            (gen, tree.get_number_of_branches(), tree.get_total_leaves(), tree.get_reserve())
        )


tree = grow_tree(cfg, n_generations=n_generations, seed=42, on_step=on_step)

print(f"{'gen':>6}  {'branches':>10}  {'leaves':>8}  {'reserve':>10}")
for gen, nb, nl, r in history:
    print(f"{gen:>6}  {nb:>10}  {nl:>8}  {r:>10.4f}")
print(
    f"\nFinal: {tree.get_number_of_branches()} branches, "
    f"{tree.get_total_leaves()} leaves, trunk diameter = {tree.get_diameter(0):.4f} m"
)

Notice the *non-monotonic* branch count: the tree adds twigs, then a
stronger-than-average storm prunes the most over-extended limbs, and
growth restarts. This wind-driven self-pruning is the central mechanism
behind the self-similar shapes in the paper.

## Visualise the canopy

`plot_tree_3d` returns a plotly figure. Each branch is coloured by its
Strahler order — trunk in dark green, twigs in pale yellow.

In [ ]:
fig = plot_tree_3d(tree)
fig.show()

## Overlay the leaves

The same plot can show the leaf cloud, sized by absorbed light. We need
to pass the `Sun` model so colours reflect the actual light absorption.

In [ ]:
sun = Sun(
    n_elevations=cfg.light.n_elevations,
    n_azimuths=cfg.light.n_azimuths,
    size_leaf=cfg.light.size_leaf,
)
fig = plot_tree_3d(tree, show_leaves=True, sun=sun)
fig.show()

Leaves near the top intercept more light — the inner canopy is shaded
out. That asymmetry is what drives phototropic growth in the next round.

## Where to next

- [02_forest_under_wind.ipynb](02_forest_under_wind.ipynb) — many trees
  competing for light, with self-thinning.
- [04_custom_growth_law.ipynb](04_custom_growth_law.ipynb) — plug in
  your own wind / sun / genome.
- User guide: [Simulating a single tree](../docs/userguide.rst).